# 🔍 Explorative Datenanalyse - Help Desk Performance

Notebook für initiale Datenexploration des Help Desk Performance Appraisal Systems.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisierung
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Pfade
DATA_DIR = Path("../data/raw")
REPORTS_DIR = Path("../reports")

print("✅ Setup abgeschlossen")

## 2. Daten laden

In [ ]:
print("📁 Lade Datensätze...")

# Alle Datasets laden
issues = pd.read_csv(DATA_DIR / "issues.csv")
history = pd.read_csv(DATA_DIR / "issues_change_history.csv")
snapshots = pd.read_csv(DATA_DIR / "issues_snapshot.csv")
scored = pd.read_excel(DATA_DIR / "issues_snapshot_sample.xlsx")
utterances = pd.read_csv(DATA_DIR / "sample_utterances.csv")

# Übersicht
datasets = {
    'issues': issues,
    'history': history,
    'snapshots': snapshots,
    'scored': scored,
    'utterances': utterances
}

for name, df in datasets.items():
    print(f"   ✅ {name}: {len(df):,} Zeilen, {len(df.columns)} Spalten")

## 3. Issues Analyse

In [ ]:
print("📊 ISSUES ANALYSE")
print("="*60)

print(f"\nTotal Tickets: {len(issues):,}")
print(f"\nSpalten:")
print(issues.columns.tolist())

In [ ]:
# Issue Types
print("Issue Types:")
issues['issue_type'].value_counts()

In [ ]:
# Issue Priority
print("Issue Priority:")
issues['issue_priority'].value_counts()

In [ ]:
# Issue Status
print("Issue Status:")
issues['issue_status'].value_counts()

In [ ]:
# Numerische Statistiken
num_cols = ['wf_total_time', 'processing_steps', 'issue_comments_count']
issues[num_cols].describe()

## 4. Score Analyse (Ground Truth)

In [ ]:
print("🎯 SCORE ANALYSE (Ground Truth)")
print("="*60)

print(f"\nBewertete Samples: {len(scored)}")
print(f"Unique Assignees: {scored['assignee'].nunique()}")
print(f"Unique Projekte: {scored['project'].nunique()}")

In [ ]:
# Score-Verteilung
score_cols = ['Q1', 'Q2', 'Q3']

for q in score_cols:
    valid = scored[scored[q] > 0][q]
    print(f"\n{q} (nur >0):")
    print(f"   N: {len(valid)}")
    print(f"   Mean: {valid.mean():.2f}")
    print(f"   Std: {valid.std():.2f}")

In [ ]:
# Score Histogramm
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, q in enumerate(['Q1', 'Q2', 'Q3']):
    valid = scored[scored[q] > 0][q]
    axes[i].hist(valid, bins=5, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{q} Verteilung')
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Anzahl')
    axes[i].axvline(valid.mean(), color='red', linestyle='--', label=f'Mean: {valid.mean():.2f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Korrelationsmatrix
print("🔗 Score-Korrelationen:")
corr = scored[score_cols].corr()
print(corr.round(3))

# Visualisierung
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Score-Korrelationsmatrix')
plt.show()

In [ ]:
# Halo-Effekt Check
avg_corr = (corr.values.sum() - 3) / 6
print(f"\nDurchschnittliche Inter-Korrelation: {avg_corr:.3f}")

if avg_corr > 0.8:
    print("⚠️ WARNUNG: Hohe Inter-Korrelation!")
    print("   → Möglicher HALO-EFFEKT in den Bewertungen!")
else:
    print("✅ Korrelation im akzeptablen Bereich")

## 5. Utterances Analyse (NLP)

In [ ]:
print("💬 UTTERANCES ANALYSE")
print("="*60)

print(f"\nTotal Utterances: {len(utterances):,}")
print(f"\nSpalten: {utterances.columns.tolist()}")

In [ ]:
# Erste Zeilen
utterances.head()

In [ ]:
# Author Roles
if 'author_role' in utterances.columns:
    print("Author Roles:")
    print(utterances['author_role'].value_counts())

## 6. Zusammenfassung

In [ ]:
print("="*60)
print("📋 ZUSAMMENFASSUNG")
print("="*60)

print(f"\n✅ Datensätze geladen:")
print(f"   - {len(issues):,} Tickets")
print(f"   - {len(scored):,} bewertete Samples (Ground Truth)")
print(f"   - {len(utterances):,} Utterances (NLP)")
print(f"   - {scored['assignee'].nunique()} einzigartige Mitarbeiter")

print(f"\n⚠️ Erkannte Probleme:")
print(f"   - Starker Halo-Effekt in Scores (Korr: {avg_corr:.3f})")

print(f"\n🚀 Nächste Schritte:")
print(f"   1. Feature Engineering")
print(f"   2. ML-Modell Training")
print(f"   3. Objektivitätsprüfung")